In [0]:
dbutils.widgets.text("catalog", "claims", "Catalog Name")
catalog = dbutils.widgets.get("catalog")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS IDENTIFIER(:catalog || '.analytics.dim_date') (
    date_key INT,
    full_date DATE,
    year INT,
    quarter INT,
    month INT,
    month_start DATE,
    week_start DATE
);

In [0]:
%sql
INSERT OVERWRITE TABLE IDENTIFIER(:catalog || '.analytics.dim_date') 

SELECT
    CAST(DATE_FORMAT(calendar_date, 'yyyyMMdd') AS INT) AS date_key,

    calendar_date AS full_date,

    YEAR(calendar_date) AS year,

    QUARTER(calendar_date) AS quarter,

    MONTH(calendar_date) AS month,

    TRUNC(calendar_date, 'MM') AS month_start,

    DATE_SUB(
        calendar_date,
        DAYOFWEEK(calendar_date) - 2
    ) AS week_start

FROM (
    SELECT EXPLODE(
        SEQUENCE(
            DATE '2025-07-01',
            DATE '2026-06-30',
            INTERVAL 1 DAY
        )
    ) AS calendar_date
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS IDENTIFIER(:catalog || ".analytics.dim_member") (
    member_sk BIGINT,
    member_id STRING,
    gender STRING,
    city STRING,
    state STRING
);

In [0]:
%sql
INSERT OVERWRITE TABLE IDENTIFIER(:catalog || ".analytics.dim_member")

SELECT
    ROW_NUMBER() OVER (ORDER BY member_id) AS member_sk,
    member_id,
    gender,
    city,
    state
FROM (
    SELECT DISTINCT
        member_id,
        gender,
        city,
        state
    FROM claims.raw.members
    WHERE member_id IS NOT NULL
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS IDENTIFIER(:catalog || ".analytics.dim_hospital") (
    hospital_sk BIGINT,
    hospital_id STRING,
    name STRING,
    city STRING,
    city_tier STRING,
    state STRING
);

In [0]:
%sql
INSERT OVERWRITE TABLE IDENTIFIER(:catalog || ".analytics.dim_hospital")

SELECT
    ROW_NUMBER() OVER (ORDER BY hospital_id) AS hospital_sk,
    hospital_id,
    name,
    city,
    city_tier,
    state
FROM (
    SELECT DISTINCT
        hospital_id,
        name,
        city,
        city_tier,
        state
    FROM claims.raw.hospitals
    WHERE hospital_id IS NOT NULL
);

In [0]:
%sql
CREATE OR REPLACE TABLE claims.analytics.dim_plan AS

SELECT
    ROW_NUMBER() OVER (
        ORDER BY p.member_id, p.policy_id, p.__START_AT
    ) AS plan_sk,

    p.member_id,
    p.policy_id,
    p.plan_code,

    pl.plan_family,

    CAST(pl.sum_insured AS INT) AS sum_insured,
    CAST(pl.room_rent_cap_per_day AS INT) AS room_rent_cap_per_day,
    CAST(pl.annual_premium AS INT) AS annual_premium,

    CAST(p.__START_AT AS DATE) AS valid_from,
    CAST(p.__END_AT AS DATE) AS valid_to,

    CASE
        WHEN p.__END_AT IS NULL THEN TRUE
        ELSE FALSE
    END AS is_current

FROM claims.silver.policy_scd2 p

INNER JOIN claims.raw.plans pl
    ON p.plan_code = pl.plan_code;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS claims.analytics.fact_claims (
    claim_id STRING,
    member_sk BIGINT,
    hospital_sk BIGINT,
    plan_sk BIGINT,
    admission_date_key INT,
    submitted_date_key INT,
    settled_date_key INT,
    claim_type STRING,
    status STRING,
    rejection_code STRING,
    is_network_at_admission BOOLEAN,
    room_rent_over_cap BOOLEAN,
    billed_amount DECIMAL(18,2),
    approved_amount DECIMAL(18,2),
    length_of_stay INT,
    turnaround_days INT
);

In [0]:
%sql
INSERT OVERWRITE TABLE claims.analytics.fact_claims

SELECT
    c.claim_id,

    m.member_sk,
    h.hospital_sk,
    p.plan_sk,

    CAST(DATE_FORMAT(c.admission_date, 'yyyyMMdd') AS INT)
        AS admission_date_key,

    CAST(DATE_FORMAT(TO_DATE(c.submitted_ts), 'yyyyMMdd') AS INT)
        AS submitted_date_key,

    CASE
        WHEN c.settled_ts IS NOT NULL
        THEN CAST(DATE_FORMAT(TO_DATE(c.settled_ts), 'yyyyMMdd') AS INT)
        ELSE NULL
    END AS settled_date_key,

    c.claim_type,
    c.status,
    c.rejection_code,

    c.is_network_at_admission,
    c.room_rent_over_cap,

    CAST(c.billed_amount AS DECIMAL(18,2)) AS billed_amount,

    CAST(c.approved_amount AS DECIMAL(18,2)) AS approved_amount,

    GREATEST(
        DATEDIFF(c.discharge_date, c.admission_date),
        1
    ) AS length_of_stay,

    CASE
        WHEN c.settled_ts IS NOT NULL
        THEN DATEDIFF(
            TO_DATE(c.settled_ts),
            TO_DATE(c.submitted_ts)
        )
        ELSE NULL
    END AS turnaround_days

FROM claims.gold.claims_enriched c

LEFT JOIN claims.analytics.dim_member m
    ON c.member_id = m.member_id

LEFT JOIN claims.analytics.dim_hospital h
    ON c.hospital_id = h.hospital_id

LEFT JOIN claims.analytics.dim_plan p
    ON c.policy_id = p.policy_id
    AND TO_DATE(c.admission_date) >= p.valid_from
    AND (
        p.valid_to IS NULL
        OR TO_DATE(c.admission_date) <= p.valid_to
    );

In [0]:
%sql
ALTER TABLE claims.analytics.dim_date
SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name');

ALTER TABLE claims.analytics.dim_date
RENAME COLUMN date_key TO dim_date_pk;

ALTER TABLE claims.analytics.dim_member
SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name');

ALTER TABLE claims.analytics.dim_member
RENAME COLUMN member_sk TO dim_member_pk;

ALTER TABLE claims.analytics.dim_hospital
SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name');

ALTER TABLE claims.analytics.dim_hospital
RENAME COLUMN hospital_sk TO dim_hospital_pk;

ALTER TABLE claims.analytics.dim_plan
SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name');

ALTER TABLE claims.analytics.dim_plan
RENAME COLUMN plan_sk TO dim_plan_pk;

In [0]:
%sql
ALTER TABLE claims.analytics.fact_claims
SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name');

ALTER TABLE claims.analytics.fact_claims
RENAME COLUMN member_sk TO fact_claims_member_fk;

ALTER TABLE claims.analytics.fact_claims
RENAME COLUMN hospital_sk TO fact_claims_hospital_fk;

ALTER TABLE claims.analytics.fact_claims
RENAME COLUMN plan_sk TO fact_claims_plan_fk;

ALTER TABLE claims.analytics.fact_claims
RENAME COLUMN admission_date_key TO fact_claims_admission_date_fk;

ALTER TABLE claims.analytics.fact_claims
RENAME COLUMN submitted_date_key TO fact_claims_submitted_date_fk;

ALTER TABLE claims.analytics.fact_claims
RENAME COLUMN settled_date_key TO fact_claims_settled_date_fk;

In [0]:
%sql
ALTER TABLE IDENTIFIER(:catalog || ".analytics.fact_claims")
CLUSTER BY (fact_claims_admission_date_fk);

### R2 - SQL editor analytics

In [0]:
%sql
-- Q01 — Claims count by status
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || ".analytics.q01") AS
SELECT
    status,
    CAST(COUNT(*) AS INT) AS claim_count
FROM claims.analytics.fact_claims
GROUP BY status;

In [0]:
%sql
-- Q02 — Top 10 hospitals by payments
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || ".analytics.q02") AS

SELECT
    h.hospital_id,
    h.name,
    CAST(
        SUM(f.approved_amount) AS DECIMAL(18,2)
    ) AS total_paid,
    CAST(COUNT(*) AS INT) AS settled_claims
FROM claims.analytics.fact_claims f

INNER JOIN claims.analytics.dim_hospital h
    ON f.fact_claims_hospital_fk = h.dim_hospital_pk
WHERE LOWER(TRIM(f.status)) = 'settled'

GROUP BY
    h.hospital_id,
    h.name
ORDER BY total_paid DESC

LIMIT 10;

In [0]:
%sql
-- Q03 — Monthly claim and billing trends
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || ".analytics.q03") AS

SELECT
    d.month_start,
    CAST(COUNT(*) AS INT) AS claim_count,
    CAST(
        SUM(f.billed_amount) AS DECIMAL(18,2)
    ) AS total_billed
FROM claims.analytics.fact_claims f

INNER JOIN claims.analytics.dim_date d
    ON f.fact_claims_admission_date_fk = d.dim_date_pk
GROUP BY d.month_start
ORDER BY d.month_start;

In [0]:
%sql
-- Q04 — Settlement turnaround analysis
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || ".analytics.q04") AS

SELECT
    claim_type,
    is_network_at_admission,
    CAST(COUNT(*) AS INT) AS settled_claims,
    CAST(
        AVG(turnaround_days) AS DECIMAL(18,2)
    ) AS avg_turnaround_days
FROM claims.analytics.fact_claims

WHERE LOWER(TRIM(status)) = 'settled'
GROUP BY
    claim_type,
    is_network_at_admission
ORDER BY
    claim_type,
    is_network_at_admission;

In [0]:
%sql
-- Q05 — Rejection code analysis
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || ".analytics.q05") AS

WITH rejection_counts AS (
    SELECT
        COALESCE(rejection_code, 'UNKNOWN') AS rejection_code,
        COUNT(*) AS rejected_count
    FROM claims.analytics.fact_claims
    WHERE LOWER(TRIM(status)) = 'rejected'
    GROUP BY COALESCE(rejection_code, 'UNKNOWN')
)

SELECT
    rejection_code,
    CAST(rejected_count AS INT) AS rejected_count,
    CAST(
        rejected_count * 1.0 /
        SUM(rejected_count) OVER ()
        AS DECIMAL(10,4)
    ) AS share_of_rejections

FROM rejection_counts
ORDER BY rejected_count DESC;

In [0]:
%sql
-- Q06 — Room rent cap analysis by plan
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || ".analytics.q06") AS

SELECT
    p.plan_code,
    CAST(COUNT(*) AS INT) AS claim_count,
    CAST(
        SUM(
            CASE
                WHEN f.room_rent_over_cap = TRUE THEN 1
                ELSE 0
            END
        ) AS INT
    ) AS over_cap_count,
    CAST(
        SUM(
            CASE
                WHEN f.room_rent_over_cap = TRUE THEN 1
                ELSE 0
            END
        ) * 1.0 / COUNT(*)
        AS DECIMAL(10,4)
    ) AS over_cap_share
FROM claims.analytics.fact_claims f

INNER JOIN claims.analytics.dim_plan p
    ON f.fact_claims_plan_fk = p.dim_plan_pk
GROUP BY p.plan_code
ORDER BY over_cap_share DESC;

### R3 - Window functions + cohorts

In [0]:
%sql
-- 1. Create W01
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || ".analytics.w01") AS

WITH months AS (
    SELECT DISTINCT
        month_start,
        LAST_DAY(month_start) AS month_end,
        DAY(LAST_DAY(month_start)) AS days_in_month
    FROM claims.analytics.dim_date
),

plan_months AS (
    SELECT
        p.plan_code,
        m.month_start,
        SUM(
            CAST(p.annual_premium AS DECIMAL(18,2))
            / 12
            *
            (
                DATEDIFF(
                    LEAST(
                        COALESCE(p.valid_to, m.month_end),
                        m.month_end
                    ),
                    GREATEST(p.valid_from, m.month_start)
                ) + 1
            )
            / m.days_in_month
        ) AS earned_premium
    FROM claims.analytics.dim_plan p
    CROSS JOIN months m
    WHERE p.valid_from <= m.month_end
      AND COALESCE(p.valid_to, m.month_end) >= m.month_start
    GROUP BY p.plan_code, m.month_start
),

paid AS (
    SELECT
        p.plan_code,
        d.month_start,
        SUM(f.approved_amount) AS paid_amount
    FROM claims.analytics.fact_claims f
    INNER JOIN claims.analytics.dim_plan p
        ON f.fact_claims_plan_fk = p.dim_plan_pk
    INNER JOIN claims.analytics.dim_date d
        ON f.fact_claims_settled_date_fk = d.dim_date_pk
    WHERE LOWER(TRIM(f.status)) = 'settled'
    GROUP BY p.plan_code, d.month_start
)

SELECT
    pm.plan_code,
    pm.month_start,
    CAST(COALESCE(pa.paid_amount, 0) AS DECIMAL(18,2))
        AS paid_amount,
    CAST(pm.earned_premium AS DECIMAL(18,2))
        AS earned_premium,
    CAST(
        COALESCE(pa.paid_amount, 0) / NULLIF(pm.earned_premium, 0)
        AS DECIMAL(18,2)
    ) AS loss_ratio
FROM plan_months pm
LEFT JOIN paid pa
    ON pm.plan_code = pa.plan_code
   AND pm.month_start = pa.month_start;

In [0]:
%sql
-- W02 — Rolling three-month loss ratio
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || ".analytics.w02") AS

SELECT
    plan_code,
    month_start,
    CAST(loss_ratio AS DECIMAL(18,2)) AS loss_ratio,
    CAST(
        SUM(paid_amount) OVER (
            PARTITION BY plan_code
            ORDER BY month_start
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        )
        /
        NULLIF(
            SUM(earned_premium) OVER (
                PARTITION BY plan_code
                ORDER BY month_start
                ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
            ),
            0
        )
        AS DECIMAL(18,2)
    ) AS rolling_3m_loss_ratio
FROM claims.analytics.w01;

In [0]:
%sql
-- W03 — Hospital ranking within city tier
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || ".analytics.w03") AS

WITH hospital_payments AS (
    SELECT
        h.hospital_id,
        h.city_tier,
        CAST(
            SUM(f.approved_amount) AS DECIMAL(18,2)
        ) AS total_paid
    FROM claims.analytics.fact_claims f
    INNER JOIN claims.analytics.dim_hospital h
        ON f.fact_claims_hospital_fk = h.dim_hospital_pk
    WHERE LOWER(TRIM(f.status)) = 'settled'
    GROUP BY
        h.hospital_id,
        h.city_tier
)

SELECT
    hospital_id,
    city_tier,
    total_paid,
    CAST(
        RANK() OVER (
            PARTITION BY city_tier
            ORDER BY total_paid DESC
        ) AS INT
    ) AS rank_in_tier
FROM hospital_payments;

In [0]:
%sql
-- W04 — Member cohort analysis
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || ".analytics.w04") AS

WITH member_cohorts AS (
    SELECT
        member_id,
        MIN(valid_from) AS first_policy_date,
        CONCAT(
            YEAR(MIN(valid_from)),
            '-Q',
            QUARTER(MIN(valid_from))
        ) AS cohort_quarter
    FROM claims.analytics.dim_plan
    GROUP BY member_id
),

member_claims AS (
    SELECT
        m.member_id,
        COUNT(f.claim_id) AS claim_count
    FROM claims.analytics.dim_member m
    LEFT JOIN claims.analytics.fact_claims f
        ON m.dim_member_pk = f.fact_claims_member_fk
    GROUP BY m.member_id
),

cohort_summary AS (
    SELECT
        mc.cohort_quarter,
        COUNT(*) AS members,
        COUNT(
            CASE
                WHEN COALESCE(cl.claim_count, 0) > 0
                THEN 1
            END
        ) AS claimants,
        SUM(
            COALESCE(cl.claim_count, 0)
        ) AS claim_count
    FROM member_cohorts mc
    LEFT JOIN member_claims cl
        ON mc.member_id = cl.member_id
    GROUP BY mc.cohort_quarter
)

SELECT
    cohort_quarter,
    CAST(members AS INT) AS members,
    CAST(claimants AS INT) AS claimants,
    CAST(claim_count AS INT) AS claim_count,
    CAST(
        claim_count * 1.0 / NULLIF(members, 0)
        AS DECIMAL(10,4)
    ) AS claims_per_member,
    CAST(
        SUM(members) OVER (
            ORDER BY cohort_quarter
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS INT
    ) AS cumulative_members
FROM cohort_summary;

In [0]:
%sql
CREATE OR REPLACE VIEW IDENTIFIER(:catalog || ".analytics.mv_claims")
WITH METRICS LANGUAGE YAML AS
$$

version: 1.1

comment: "Centralized claims metrics for business reporting and analytics."

source: claims.analytics.fact_claims

joins:

  - name: plan
    source: claims.analytics.dim_plan
    'on': source.fact_claims_plan_fk = plan.dim_plan_pk
    rely:
      at_most_one_match: true

  - name: hospital
    source: claims.analytics.dim_hospital
    'on': source.fact_claims_hospital_fk = hospital.dim_hospital_pk
    rely:
      at_most_one_match: true

  - name: admission_date
    source: claims.analytics.dim_date
    'on': source.fact_claims_admission_date_fk = admission_date.dim_date_pk
    rely:
      at_most_one_match: true

fields:

  - name: plan_code
    expr: plan.plan_code

  - name: city_tier
    expr: hospital.city_tier

  - name: hospital_id
    expr: hospital.hospital_id

  - name: claim_type
    expr: source.claim_type

  - name: admission_month
    expr: admission_date.month_start

measures:

  - name: claim_count
    expr: COUNT(DISTINCT source.claim_id)
    comment: "Total distinct claims."

  - name: approved_amount
    expr: SUM(source.approved_amount)
    comment: "Total approved claim amount."

  - name: rejection_rate
    expr: |
      CAST(
        SUM(CASE
          WHEN LOWER(TRIM(source.status)) = 'rejected'
          THEN 1 ELSE 0
        END) AS DOUBLE
      )
      /
      NULLIF(
        SUM(CASE
          WHEN LOWER(TRIM(source.status))
               IN ('approved', 'rejected', 'settled')
          THEN 1 ELSE 0
        END),
        0
      )
    comment: "Rejected claims divided by decided claims. Pending claims are excluded."

  - name: avg_turnaround_days
    expr: AVG(source.turnaround_days)
    comment: "Average settlement turnaround days."

$$;

In [0]:
%sql
CREATE OR REPLACE VIEW IDENTIFIER(:catalog || ".analytics.mv_loss_ratio")
WITH METRICS LANGUAGE YAML AS
$$

version: 1.1

comment: "Centralized monthly loss ratio metrics by insurance plan."

source: claims.analytics.w01

fields:

  - name: plan_code
    expr: plan_code

  - name: month_start
    expr: month_start

measures:

  - name: paid_amount
    expr: SUM(paid_amount)
    comment: "Total paid claim amount."

  - name: earned_premium
    expr: SUM(earned_premium)
    comment: "Total earned premium."

  - name: loss_ratio
    expr: |
      SUM(paid_amount) /
      NULLIF(SUM(earned_premium), 0)
    comment: "Total paid amount divided by total earned premium."

$$;

In [0]:
%sql
-- Create claims.analytics.metric_definitions
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || ".analytics.metric_definitions") (
    metric_name STRING,
    expression STRING,
    source STRING,
    description STRING
);

In [0]:
%sql
-- Insert exactly five metric definitions
INSERT OVERWRITE TABLE IDENTIFIER(:catalog || ".analytics.metric_definitions")

SELECT *
FROM VALUES

(
    'claim_count',
    'COUNT(DISTINCT claim_id)',
    'claims.analytics.fact_claims',
    'Total number of distinct claims.'
),

(
    'approved_amount',
    'SUM(approved_amount)',
    'claims.analytics.fact_claims',
    'Total approved claim amount.'
),

(
    'rejection_rate',
    'Rejected claims / decided claims',
    'claims.analytics.fact_claims',
    'Rejected claims divided by approved, rejected and settled claims. Pending claims are excluded.'
),

(
    'avg_turnaround_days',
    'AVG(turnaround_days)',
    'claims.analytics.fact_claims',
    'Average settlement turnaround time in days.'
),

(
    'loss_ratio',
    'SUM(paid_amount) / SUM(earned_premium)',
    'claims.analytics.w01',
    'Total paid amount divided by total earned premium.'
)

AS t(metric_name, expression, source, description);

In [0]:
%sql
-- Create v_metrics_by_plan
CREATE OR REPLACE VIEW IDENTIFIER(:catalog || ".analytics.v_metrics_by_plan") AS

WITH claims_metrics AS (

    SELECT
        p.plan_code,

        CAST(
            COUNT(DISTINCT f.claim_id) AS INT
        ) AS claim_count,

        CAST(
            SUM(f.approved_amount) AS DECIMAL(18,2)
        ) AS approved_amount,

        CAST(
            SUM(
                CASE
                    WHEN LOWER(TRIM(f.status)) = 'rejected'
                    THEN 1 ELSE 0
                END
            ) * 1.0
            /
            NULLIF(
                SUM(
                    CASE
                        WHEN LOWER(TRIM(f.status))
                             IN ('approved', 'rejected', 'settled')
                        THEN 1 ELSE 0
                    END
                ),
                0
            )
            AS DECIMAL(10,4)
        ) AS rejection_rate,

        CAST(
            AVG(f.turnaround_days) AS DECIMAL(18,2)
        ) AS avg_turnaround_days

    FROM claims.analytics.fact_claims f

    INNER JOIN claims.analytics.dim_plan p
        ON f.fact_claims_plan_fk = p.dim_plan_pk

    GROUP BY p.plan_code
),

loss_metrics AS (

    SELECT
        plan_code,

        CAST(
            SUM(paid_amount) / NULLIF(SUM(earned_premium), 0)
            AS DECIMAL(18,2)
        ) AS loss_ratio

    FROM claims.analytics.w01

    GROUP BY plan_code
)

SELECT
    c.plan_code,
    c.claim_count,
    c.approved_amount,
    c.rejection_rate,
    c.avg_turnaround_days,
    l.loss_ratio

FROM claims_metrics c

LEFT JOIN loss_metrics l
    ON c.plan_code = l.plan_code;

In [0]:
%sql
select status from claims.analytics.fact_claims group by status

In [0]:
%sql
-- Create claims.analytics.weekly_rejection_rate
CREATE OR REPLACE TABLE claims.analytics.weekly_rejection_rate AS
SELECT
    dp.plan_code,
    dd.week_start,
    CAST(
        SUM(
            CASE
                WHEN LOWER(TRIM(fc.status))
                     IN ('approved', 'rejected', 'settled')
                THEN 1 ELSE 0
            END
        ) AS INT
    ) AS decided_count,
    CAST(
        SUM(
            CASE
                WHEN LOWER(TRIM(fc.status)) = 'rejected'
                THEN 1 ELSE 0
            END
        ) AS INT
    ) AS rejected_count,
    CAST(
        SUM(
            CASE
                WHEN LOWER(TRIM(fc.status)) = 'rejected'
                THEN 1.0 ELSE 0.0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN LOWER(TRIM(fc.status))
                         IN ('approved', 'rejected', 'settled')
                    THEN 1 ELSE 0
                END
            ),
            0
        )
        AS DECIMAL(10,4)
    ) AS rejection_rate
FROM claims.analytics.fact_claims fc
JOIN claims.analytics.dim_plan dp
    ON fc.fact_claims_plan_fk = dp.dim_plan_pk
JOIN claims.analytics.dim_date dd
    ON fc.fact_claims_admission_date_fk = dd.dim_date_pk
GROUP BY
    dp.plan_code,
    dd.week_start;

In [0]:
%sql
SELECT *
FROM claims.analytics.weekly_rejection_rate
ORDER BY week_start, plan_code;

In [0]:
%sql
SELECT
    MAX(rejection_rate) AS highest_plan_rejection_rate
FROM claims.analytics.weekly_rejection_rate
WHERE week_start = DATE '2026-06-22';

In [0]:
%sql
SELECT COUNT(*)
FROM claims_Lakebase.public.provider_directory;

In [0]:
%sql
SELECT
    hospital_id,
    accreditation,
    accredited_until
FROM claims_Lakebase.public.provider_directory
LIMIT 10;

In [0]:
%sql

CREATE OR REPLACE TABLE claims.analytics.`8_provider_jo` AS

SELECT
    dh.hospital_id,
    dh.name,
    dh.city_tier,
    pd.accreditation,
    pd.accredited_until,

    CAST(
        COUNT(DISTINCT
            CASE
                WHEN pd.accredited_until IS NOT NULL
                     AND dd.full_date > pd.accredited_until
                THEN fc.claim_id
            END
        ) AS INT
    ) AS claims_after_lapse

FROM claims.analytics.dim_hospital dh

LEFT JOIN claims_Lakebase.public.provider_directory pd
    ON dh.hospital_id = pd.hospital_id

LEFT JOIN claims.analytics.fact_claims fc
    ON dh.dim_hospital_pk = fc.fact_claims_hospital_fk

LEFT JOIN claims.analytics.dim_date dd
    ON fc.fact_claims_admission_date_fk = dd.dim_date_pk

GROUP BY
    dh.hospital_id,
    dh.name,
    dh.city_tier,
    pd.accreditation,
    pd.accredited_until;

In [0]:
%sql
SELECT *
FROM claims.analytics.`8_provider_jo`
ORDER BY claims_after_lapse DESC;

In [0]:
%sql
DESCRIBE TABLE claims.analytics.`8_provider_jo`;

In [0]:
%sql

CREATE OR REPLACE TABLE claims.analytics.r9_paradox AS

WITH base AS (
    SELECT
        dh.hospital_id,
        LOWER(TRIM(fc.claim_type)) AS claim_type,
        fc.claim_id,
        LOWER(TRIM(fc.status)) AS status
    FROM claims.analytics.fact_claims fc
    JOIN claims.analytics.dim_hospital dh
        ON fc.fact_claims_hospital_fk = dh.dim_hospital_pk
    WHERE dh.hospital_id IN ('HOS-017', 'HOS-022')
),

decided AS (
    SELECT *
    FROM base
    WHERE status IN ('approved', 'rejected', 'settled')
),

overall AS (
    SELECT
        hospital_id,
        'ALL' AS claim_type,
        COUNT(DISTINCT claim_id) AS decided_count,
        COUNT(DISTINCT CASE
            WHEN status = 'rejected' THEN claim_id
        END) AS rejected_count
    FROM decided
    GROUP BY hospital_id
),

by_claim_type AS (
    SELECT
        hospital_id,
        claim_type,
        COUNT(DISTINCT claim_id) AS decided_count,
        COUNT(DISTINCT CASE
            WHEN status = 'rejected' THEN claim_id
        END) AS rejected_count
    FROM decided
    WHERE claim_type IN ('cashless', 'reimbursement')
    GROUP BY hospital_id, claim_type
),

combined AS (
    SELECT * FROM overall
    UNION ALL
    SELECT * FROM by_claim_type
)

SELECT
    hospital_id,
    claim_type,
    CAST(decided_count AS INT) AS decided_count,
    CAST(rejected_count AS INT) AS rejected_count,
    CAST(
        rejected_count * 1.0 / NULLIF(decided_count, 0)
        AS DECIMAL(10,4)
    ) AS rejection_rate
FROM combined;

In [0]:
%sql
SELECT *
FROM claims.analytics.r9_paradox
ORDER BY
    CASE claim_type
        WHEN 'ALL' THEN 1
        WHEN 'cashless' THEN 2
        WHEN 'reimbursement' THEN 3
    END,
    hospital_id;